# A Thai character-level GPT, for the Bayesian stopping controller's LM prior

> **Run this on Google Colab (GPU runtime), not locally.** This notebook now downloads the full
> ~430MB / ~450M-character corpus and trains a bigger model than the first pass did — appropriate
> for a Colab GPU, not a laptop. `Runtime -> Change runtime type -> GPU` before running top to
> bottom. The corpus/model detects Colab automatically and adjusts its paths accordingly (see
> Setup).

Inspired by Andrej Karpathy's ["Let's build GPT: from scratch, in code, spelled out"](https://www.youtube.com/watch?v=kCc8FmEb1nY) —
same architecture and teaching progression (a bigram baseline's assumptions, then self-attention,
multi-head attention, transformer blocks), adapted to:

- **Thai text** instead of Shakespeare (`pythainlp/thai-wiki-dataset-v3`, a Thai Wikipedia dump).
- **A fixed 72-character output alphabet** — this project's speller matrix
  (`collection/characters_th.json`), not the model's full natural-language vocabulary. The model
  trains on *real* unrestricted Thai text (spaces, rare consonants, foreign scripts and all) so it
  learns real orthographic structure, then at serving time we take its full prediction and
  **marginalize** it down to just the 72 matrix characters. That's the one piece of logic here
  that isn't in Karpathy's original tutorial, and it's the part that actually matters for this
  project.

**Why this started at 10% of the data on a small model, and only now scales up**: the first pass
(0.88M params, 25M characters, trained locally in 3 minutes) existed to prove the whole path was
*correct* — real data in, a working tokenizer, a real transformer, a properly marginalized
`get_prior()`, a saved checkpoint — before spending real compute on it. It worked (verified:
`get_prior("ประเทศไ")` put 93.6% on the correct next character), so now it's worth scaling up. Cheap
mistakes are worth catching small; this config is worth catching on a GPU, not a laptop.

This notebook trains the model and proves the whole path end to end: real data → tokenizer →
transformer → a working `get_prior(context) -> 72-length distribution` → a saved checkpoint. Wiring
that checkpoint into `pipeline/language_model.py` as a `"transformer"` LM type (alongside the
existing `"uniform"`/`"ngram"` types) is deliberately **not** done here — see the last section.

## Setup

In [ ]:
import json
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = None  # not checked out on Colab -- collection/characters_th.json is uploaded below instead
    DATA_DIR = "/content/thai_gpt_data"
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # stopping/ -> repo root
    DATA_DIR = os.path.join(REPO_ROOT, "stopping", "data")
os.makedirs(DATA_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, "| Colab:", IN_COLAB)
if not IN_COLAB and device == "cpu":
    print("WARNING: this config (full corpus, bigger model) is sized for a Colab GPU runtime.")
    print("Running it locally on CPU would be very slow -- see this notebook's first cell.")

## 1. Data: the full Thai Wikipedia corpus

`pythainlp/thai-wiki-dataset-v3` on HuggingFace — Wikipedia + Wikiquote + Wikibooks + Wikisource +
Wiktionary, CC-BY-SA-3.0. It ships as 3 parquet shards (220MB + 122MB + 88MB ≈ 430MB total,
196,533 articles). The first pass used one shard, subsampled to 25M characters, specifically to
prove the pipeline correctness cheaply; now that it's proven, this cell downloads **all three
shards and uses every article** — no subsampling.

This is a real download (~430MB) and a real amount of RAM to hold the concatenated text — this is
the main reason this version belongs on Colab, not a laptop. Downloads once and caches to
`thai_wiki_full.txt` (a different filename from the first pass's `thai_wiki_sample.txt`, so the two
don't collide and re-running this notebook won't silently reuse the smaller sample).

In [ ]:
CORPUS_PATH = os.path.join(DATA_DIR, "thai_wiki_full.txt")
SHARD_URLS = [
    "https://huggingface.co/api/datasets/pythainlp/thai-wiki-dataset-v3/parquet/default/train/0.parquet",
    "https://huggingface.co/api/datasets/pythainlp/thai-wiki-dataset-v3/parquet/default/train/1.parquet",
    "https://huggingface.co/api/datasets/pythainlp/thai-wiki-dataset-v3/parquet/default/train/2.parquet",
]

if not os.path.exists(CORPUS_PATH):
    print(f"downloading all {len(SHARD_URLS)} shards of pythainlp/thai-wiki-dataset-v3 (~430MB total) ...")
    articles = []
    for i, url in enumerate(SHARD_URLS):
        print(f"  shard {i}...")
        df = pd.read_parquet(url, columns=["text"])
        articles.extend(df["text"].tolist())
        del df

    rng = np.random.default_rng(1337)
    order = rng.permutation(len(articles))
    text = "\n\n".join(articles[i] for i in order)
    with open(CORPUS_PATH, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"saved {len(articles):,} articles, {len(text):,} characters -> {CORPUS_PATH}")
else:
    print(f"found cached corpus at {CORPUS_PATH}, skipping download")

text = open(CORPUS_PATH, encoding="utf-8").read()
print(f"corpus: {len(text):,} characters, {len(set(text)):,} unique characters")
print()
print(text[:300])

## 2. Tokenization — character-level, with a deliberate twist

Plain character-level tokenization (`chars = sorted(set(text))`, like the tutorial) would give a
vocabulary in the thousands here — most of it single-occurrence noise from Wikipedia's incidental
Chinese, Cyrillic, and math-symbol characters. That's wasted capacity: each of those tokens gets its
own embedding row and barely any training signal.

So: keep any character that appears at least `MIN_COUNT` times, map everything rarer to a single
`<unk>` token — standard practice for character-level LMs on noisy real-world text.

**One thing this cutoff would silently break if applied blindly**: `ฃ` (kho khuat) and `ฅ` (kho
khon) are two archaic Thai consonants, still present in `characters_th.json`'s matrix (the reference
speller's alphabet includes the full historical 44-consonant set) but essentially unused in modern
written Thai. Checking this corpus: they occur only 19 and 33 times in 25M characters — both well
under a `MIN_COUNT=100` cutoff. A naive frequency-based vocabulary would merge both into `<unk>`,
and the speller's marginalization step (next section) would then have no real slot for them at
all — every context would report a nonsensical prior for those two matrix cells.

**Fix**: force-include all 72 matrix characters in the vocabulary regardless of their corpus
frequency, on top of the frequency cutoff for everything else. Rare tokens still get poorly-informed
embeddings (there's no way around that with only 19-33 examples), but they exist as real tokens
instead of being unrecoverably absorbed into `<unk>`.

In [ ]:
if IN_COLAB:
    characters_path = "characters_th.json"
    if not os.path.exists(characters_path):
        print("This repo has no public git remote, so characters_th.json can't be cloned automatically.")
        print("Upload collection/characters_th.json from the project repo now:")
        from google.colab import files
        uploaded = files.upload()
        characters_path = next(iter(uploaded))
else:
    characters_path = os.path.join(REPO_ROOT, "collection", "characters_th.json")

matrix_chars = json.load(open(characters_path, encoding="utf-8"))["characters"]
matrix_set = set(matrix_chars)
print(f"matrix: {len(matrix_chars)} characters")

MIN_COUNT = 100
UNK = ""  # a private-use codepoint, guaranteed not to collide with real text
counts = Counter(text)
assert UNK not in counts

vocab = sorted(matrix_set | {c for c, n in counts.items() if n >= MIN_COUNT}) + [UNK]
vocab_size = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for i, c in enumerate(vocab)}
unk_id = stoi[UNK]

rare_matrix_chars = [(c, counts.get(c, 0)) for c in matrix_chars if counts.get(c, 0) < MIN_COUNT]
print(f"vocab size (incl. <unk>): {vocab_size}")
print(f"all 72 matrix characters present in vocab: {matrix_set <= set(vocab)}")
print(f"matrix characters below the frequency cutoff, force-included anyway: {rare_matrix_chars}")

def encode(s):
    return [stoi.get(c, unk_id) for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

# round-trip sanity check
sample = text[1000:1050]
assert decode(encode(sample)) == "".join(c if c in stoi else UNK for c in sample)
print("encode/decode round-trip OK")

## 3. Train/val split, and batching

90/10 split, same as the tutorial. `get_batch` samples random `block_size`-length windows and their
one-token-shifted targets — standard next-token-prediction batching.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"train: {len(train_data):,} tokens   val: {len(val_data):,} tokens")

# ---- hyperparameters ----
# Sized for a Colab GPU, not the first pass's laptop config. block_size=64 matches this
# project's real ~40-char session cap (with headroom) rather than the tutorial's 128 --
# train/serve context-length parity, and O(T^2) attention cost scales with it too.
# n_embd/n_head/n_layer scaled up together for the ~20x larger full corpus; n_head=8 keeps
# head_size=32, the same per-head width as the first pass's 128/4 config.
block_size = 64    # context window in characters
batch_size = 64
n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.1

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("one batch:", xb.shape, yb.shape)

## 4. Self-attention, from scratch

The core idea the whole tutorial builds toward: instead of every position seeing a *fixed-weight*
summary of everything before it (what a bigram model does), let each position emit a **query**
("what am I looking for?"), and let every position emit a **key** ("what do I contain?") and a
**value** ("what do I offer, if picked?"). The affinity between one position's query and every
earlier position's key becomes the attention weight.

- `wei = q @ k.T * C**-0.5` — the scaling by `1/sqrt(head_size)` keeps the dot products from growing
  with dimensionality and saturating the softmax.
- `tril` masking — position `t` can only attend to positions `<= t`. This is what makes it a
  **causal** (decoder-only, autoregressive) model: predicting the next character can't peek at it.

In [5]:
class Head(nn.Module):
    """One self-attention head."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * C ** -0.5          # (B, T, T) affinities
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # causal mask
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v                                      # weighted sum of values

## 5. Multi-head attention, and the rest of one transformer block

Several attention heads in parallel (each can specialize in a different kind of relationship —
"what mark usually follows this consonant," "what character started this word," etc.), concatenated
and projected back to `n_embd`. Then a small per-position feed-forward network, and the standard
pre-norm residual pattern (`x + sublayer(LayerNorm(x))`) that makes deep transformer stacks trainable
at all.

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Communication (attention) then computation (feed-forward), each with a residual + pre-norm."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 6. The full model

Token embedding + positional embedding, `n_layer` transformer blocks stacked, a final layer norm,
and a linear head back out to vocabulary size. `generate()` is greedy autoregressive sampling —
feed the model's own output back in as the next input, one character at a time.

In [ ]:
class ThaiGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]          # can't exceed the trained context window
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                 # only the last position's prediction
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


model = ThaiGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f}M parameters")

## 7. Training

Standard loop: AdamW, periodically estimate train/val loss (averaged over several batches, so the
printed number isn't just noisy single-batch loss). Watch that val loss tracks train loss — if it
diverges upward while train keeps falling, that's overfitting.

This config (full corpus, 6-layer/256-dim model) is sized for a Colab GPU (`T4` or better) — expect
this to be meaningfully slower per step than the first pass, offset by a real GPU instead of `mps`.
If you're seeing this print `device: cpu`, stop and switch the Colab runtime to GPU first
(`Runtime -> Change runtime type`) rather than let this run for hours on CPU.

In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=20):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
checkpoint_path = os.path.join(DATA_DIR, "thai_gpt_checkpoint.pt")

# ~20x the first pass's data and a bigger model both push against the same 5000 steps'
# diminishing-returns plateau later, not sooner -- scaled up accordingly for a Colab GPU.
max_iters = 20000
eval_interval = 1000
t0 = time.time()
for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"step {it:5d}: train {losses['train']:.4f}  val {losses['val']:.4f}  ({time.time()-t0:.1f}s)")
        # Save every eval_interval steps: a Colab disconnect mid-run then only costs back to
        # the last eval point, not the whole run. Overwrites one file rather than keeping 20
        # separate checkpoints on disk.
        torch.save({
            "model_state_dict": model.state_dict(),
            "vocab": vocab,
            "config": {
                "block_size": block_size, "n_embd": n_embd, "n_head": n_head,
                "n_layer": n_layer, "dropout": dropout, "vocab_size": vocab_size,
            },
        }, checkpoint_path)
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"done: {max_iters} steps in {time.time()-t0:.1f}s, checkpoint saved to {checkpoint_path}")

## 8. Generation sanity check

Not a rigorous evaluation — just eyeballing whether the model has learned *something* about Thai
orthography (real consonant/vowel/tone-mark clustering, plausible short function words) rather than
producing pure noise. `ประเทศไทย` ("Thailand") is a good seed: a real, common word, so a
reasonably-trained model should continue it sensibly at least for the first few characters.

In [ ]:
context = torch.tensor([encode("ประเทศไทย")], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300)[0].tolist()
print(decode(generated))

## 9. The integration piece: `get_prior()`

This is the one function that doesn't come from the tutorial — it's what makes this model usable as
`stopping/bayesian_stopping.py`'s LM prior. The contract (from `pipeline/language_model.py`):
`get_prior(context: str) -> np.ndarray`, shape `(72,)`, non-negative, summing to 1, aligned to
`characters_th.json`'s index order.

Steps:
1. Truncate `context` to the last `block_size` characters — the model has no mechanism to use
   anything further back than that.
2. Encode and run one forward pass; softmax the final position's logits over the **full** training
   vocabulary (275 symbols here, not 72).
3. **Marginalize**: pull out just the 72 matrix characters' probabilities from that full
   distribution, add a tiny epsilon (guards the pathological case where all 72 happen to round to
   exactly 0), and renormalize so they sum to 1.

This mirrors `stopping/lm_prior.py`'s `ThaiNgramLM` exactly: train on natural unrestricted text,
restrict only at the point where a prior actually needs to be produced.

In [ ]:
def get_prior(context_str, model=model, block_size=block_size):
    model.eval()
    ids = encode(context_str)[-block_size:]
    if not ids:
        ids = [unk_id]
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        logits, _ = model(x)
    full_probs = F.softmax(logits[0, -1, :], dim=-1).cpu().numpy()

    raw = np.array([full_probs[stoi[c]] for c in matrix_chars], dtype=float)
    raw = raw + 1e-8
    return raw / raw.sum()


for ctx in ["ประเทศไ", "สวัส", "ขอบ", ""]:
    p = get_prior(ctx)
    assert p.shape == (72,) and np.isclose(p.sum(), 1.0)
    top5 = np.argsort(p)[::-1][:5]
    label = ctx if ctx else "(empty context)"
    print(f"{label!r:20s} -> top-5 next: " + ", ".join(f"{matrix_chars[i]!r}={p[i]:.3f}" for i in top5))

## 10. Save the checkpoint

Bundles exactly what `get_prior()` needs to run again without this notebook: the trained weights,
the vocabulary (so `stoi`/`itos` can be rebuilt identically), and the architecture hyperparameters
(so the model class can be reconstructed with the right shapes before loading the weights into it).
Mirrors how `pipeline/classifier_io.py` bundles the classifier checkpoint + its `EpochSpec` together,
rather than shipping weights and config as separate files that could drift out of sync.

In [ ]:
checkpoint_path = os.path.join(DATA_DIR, "thai_gpt_checkpoint.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "config": {
        "block_size": block_size, "n_embd": n_embd, "n_head": n_head,
        "n_layer": n_layer, "dropout": dropout, "vocab_size": vocab_size,
    },
}, checkpoint_path)
print(f"saved checkpoint to {checkpoint_path}")

## Next steps (not done in this notebook)

This notebook proves the model trains correctly and that `get_prior()` produces a valid, properly
marginalized 72-length distribution from it. Turning this into a real `language_model` type the rest
of the system can select via the session manifest is separate, deliberately-plumbing work:

1. **`pipeline/manifest.py`** — add `"transformer"` to `LM_TYPES`, a schema branch in
   `validate_manifest_dict()` (a sha256-verified weights artifact, the same pattern as the n-gram's
   corpus), a case in `build_manifest_dict()`.
2. **`pipeline/language_model.py`** — a `_build_transformer(lm_config, manifest, characters)` that
   loads this checkpoint, reconstructs the model class from its saved `config`, and wraps it in a
   small adapter exposing `get_prior()` — registered in `LM_BUILDERS["transformer"]`. Nothing else
   in the system changes; that's the point of the factory design.
3. **Validate against real data** once available: `stopping.evaluate_recording --lm transformer` vs.
   `--lm ngram` vs. `--lm uniform` on the same recording, comparing `mean_flashes`/`itr_bits_per_min`
   directly — that comparison, not this notebook's training loss, is what actually shows whether the
   Transformer prior helps.